<a href="https://colab.research.google.com/github/psvprasad2003/DS_AI_CB/blob/main/VIDEO_SURVILENCE_IN_GROCERY_STORE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics supervision opencv-python -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 6.4 MB/s eta 0:00:00


In [ ]:
import numpy as np
import supervision as sv
from ultralytics import YOLO
import cv2
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
## Here I use the built-in GROCERY_STORE video from supervision.
#You can replace it later with your own path.
from supervision.assets import VideoAssets, download_assets
from ultralytics import YOLO  # Added this line to fix NameError

# load a small YOLO model
model = YOLO("yolov8n.pt")  # or any other YOLOv8/YOLO11 model

# input and output video paths
source_path = download_assets(VideoAssets.GROCERY_STORE)
target_path = "grocery_polygon_output.mp4"

# grab video info (width, height, fps, etc.)
video_info = sv.VideoInfo.from_video_path(source_path)
print(video_info)

[2026-06-07 04:15:07] [INFO] supervision.assets.downloader - Downloading grocery-store.mp4 assets


  0%|          | 0/103282086 [00:00<?, ?it/s]

VideoInfo(width=3840, height=2160, fps=29.97002997002997, total_frames=1002)


In [ ]:
# polygon vertices in image coordinates [x, y]
polygon = np.array([
    [1725, 1550],
    [2725, 1550],
    [3500, 2160],
    [1250, 2160]
])

# create the polygon zone object
zone = sv.PolygonZone(polygon=polygon)

# visual annotators
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=1, text_scale=0.5)

zone_annotator = sv.PolygonZoneAnnotator(
    zone=zone,
    color=sv.Color.WHITE,
    thickness=3,
    text_thickness=1,
    text_scale=0.6
)

In [ ]:
def process_frame(frame, frame_idx):
    # 1) run YOLO detection
    results = model(frame, imgsz=1280)[0]
    detections = sv.Detections.from_ultralytics(results)

    # 2) (optional) keep only "person" class, usually id 0 in COCO
    detections = detections[detections.class_id == 0]

    # 3) update polygon zone (counts how many detections are inside)
    zone.trigger(detections=detections)

    # 4) build labels: "person 0.87", etc.
    labels = [
        f"{model.names[class_id]} {confidence:0.2f}"
        for _, _, confidence, class_id, _, _ in detections
    ]

    # 5) draw boxes, labels, and polygon zone overlay
    frame = box_annotator.annotate(scene=frame, detections=detections)
    frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)
    frame = zone_annotator.annotate(scene=frame)

    return frame


In [ ]:
sv.process_video(
    source_path=source_path,
    target_path=target_path,
    callback=process_frame
)

print("Done! Saved:", target_path)



0: 736x1280 1 person, 1 bottle, 1 chair, 1 refrigerator, 77.1ms
Speed: 19.8ms preprocess, 77.1ms inference, 34.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 2 bottles, 1 chair, 1 refrigerator, 8.7ms
Speed: 17.6ms preprocess, 8.7ms inference, 1.6ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 bottle, 1 chair, 1 refrigerator, 8.7ms
Speed: 14.5ms preprocess, 8.7ms inference, 1.6ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 bottle, 1 chair, 1 refrigerator, 7.9ms
Speed: 14.3ms preprocess, 7.9ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 bottle, 1 chair, 1 refrigerator, 8.1ms
Speed: 14.7ms preprocess, 8.1ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 2 bottles, 1 chair, 1 refrigerator, 8.0ms
Speed: 14.5ms preprocess, 8.0ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 

In [ ]:
def process_video_short(source, target, max_frames=300):
    cap = cv2.VideoCapture(source)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps    = cap.get(cv2.CAP_PROP_FPS)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(target, fourcc, fps, (width, height))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx >= max_frames:
            break

        frame = process_frame(frame, frame_idx)
        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"Short video saved to: {target}")

process_video_short(source_path, "grocery_polygon_output_short.mp4", max_frames=300)



0: 736x1280 1 person, 1 bottle, 1 chair, 1 refrigerator, 9.1ms
Speed: 8.3ms preprocess, 9.1ms inference, 1.8ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 2 bottles, 1 chair, 1 refrigerator, 8.2ms
Speed: 7.6ms preprocess, 8.2ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 bottle, 1 chair, 1 refrigerator, 7.9ms
Speed: 7.4ms preprocess, 7.9ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 bottle, 1 chair, 1 refrigerator, 8.0ms
Speed: 7.5ms preprocess, 8.0ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 1 bottle, 1 chair, 1 refrigerator, 8.2ms
Speed: 7.4ms preprocess, 8.2ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 2 bottles, 1 chair, 1 refrigerator, 8.1ms
Speed: 7.4ms preprocess, 8.1ms inference, 1.5ms postprocess per image at shape (1, 3, 736, 1280)

0: 736x1280 1 person, 2 bottles

In [ ]:
from google.colab import files

files.download('/content/grocery_polygon_output.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download('/content/grocery_polygon_output_short.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download('/content/grocery-store.mp4')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>